# Joint Fit of the Circuit and the GP

K-K GP-ECM can be fitted in two ways under leave-one-state-out. The two-step fit estimates the
circuit parameters $\theta$ of each training state by least squares. It then fits the GP
hyperparameters with $\theta$ fixed. The joint fit maximizes one marginal likelihood over
$\theta$ of all training states and the GP hyperparameters together. With the output file in
`experiments/` a run takes a few seconds. Computing both fits for all 31 held-out states takes
about 41 min with `N_JOBS = 3`.

In [1]:
import os
import warnings

import pandas as pd

from eis import data, datasets, env, fitting, joint, utils

RECOMPUTE = False   # True computes every output file again
N_JOBS = 3          # number of worker processes for the fits
# folder of the output files (experiments/ in the repository root)
OUT = os.path.join(env.DIR_BASE, "experiments")
os.makedirs(OUT, exist_ok=True)

# PyTorch warns that a tensor made from a read-only NumPy array must not be written. The models
# only read these tensors. PYTHONWARNINGS passes the same filter to the worker processes.
warnings.filterwarnings("ignore", message="The given NumPy array is not writable")
os.environ["PYTHONWARNINGS"] = "ignore:The given NumPy array is not writable:UserWarning"

DATA = {name: data.load_dataset(name) for name in datasets.DATASETS}
CASES = utils.held_out_cases(DATA)

## Two-Step Fit and Joint Fit

Let $a_\mathrm{obs}$ and $b_\mathrm{obs}$ be the measured $\mathrm{Re}\,Z$ and $\mathrm{Im}\,Z$
at the training points. The residual data
$\tilde{\mathbf{y}} = (a_\mathrm{obs} - m_\mathrm{re},\ b_\mathrm{obs} - m_\mathrm{im})$ follow
$\mathcal{N}(\mathbf{0}, \mathbf{M})$ with
$\mathbf{M} = \mathbf{B}_\mathrm{tr}\mathbf{K}\mathbf{B}_\mathrm{tr}^\top + \boldsymbol{\Sigma}_\mathrm{tr}$.
The matrix $\mathbf{B} = [\mathbf{I};\ \mathbf{H}_\mathcal{G}]$ maps the real residual to both
channels, and $\mathbf{B}_\mathrm{tr}$ holds its rows at the training points. The matrix
$\boldsymbol{\Sigma}_\mathrm{tr}$ is the diagonal noise covariance. The two-step fit minimizes
the negative log marginal likelihood of $\tilde{\mathbf{y}}$ over the hyperparameters
$(\sigma_f, \ell, \sigma_{n,\mathrm{re}}, \sigma_{n,\mathrm{im}})$ with the circuit fixed. The
mean $\mathbf{m}$ depends on the circuit parameters $\theta_s$ of the training states, and so
does $\tilde{\mathbf{y}}$. The joint fit minimizes the same objective over these $\theta_s$ as
well. It starts at the two-step solution and keeps its best point. Its training negative log
marginal likelihood is therefore at most that of the two-step fit. The held-out state receives
$\theta$ interpolated from the jointly fitted parameters.

The joint fit runs Adam (300 steps, rate 0.02) and then 8 L-BFGS calls of at most 100
iterations. The dictionary `eis.joint.JOINT_FIT_KW` holds these settings. The model is
`eis.joint.JointKKGPLoso`, and `eis.joint.fit_joint_and_two_step` makes both fits for one
held-out state. In its output the columns without a prefix belong to the joint fit. The columns
with the prefix `plugin_` belong to the two-step fit.

In [2]:
def compute_joint():
    cases = [(name, s, circuit, df) for name, s, df in CASES for circuit in fitting.CIRCUITS]
    rows = utils.fit_parallel(joint.fit_joint_and_two_step, cases, N_JOBS)
    return rows.sort_values(["dataset", "circuit", "held_out_v"]).reset_index(drop=True)


fits = utils.load_or_compute(os.path.join(OUT, "04_joint_fit.csv"), compute_joint, RECOMPUTE)

[2026-10-19 06:52:00][WAR] KKGP LBFGS step aborted (_LinAlgError); keeping best params.


31 fits in 41.5 min


## Comparison

Two counts compare the fits over the 31 held-out states. The first counts the runs in which the
joint fit reaches the higher training marginal likelihood (`dmll` < 0). The second counts the
states at which the joint fit predicts the held-out points worse than the two-step fit. The
measure is the mean negative log predictive density (`nll_all` > `plugin_nll_all`). The table
gives the RMSE of both fits averaged over the states of each region.

In [3]:
fits = fits[fits["circuit"] == "ECM-W"]
print(f"Runs with a higher training marginal likelihood for the joint fit: "
      f"{int((fits['dmll'] < 0).sum())} of {len(fits)}")
print(f"States with a worse held-out log predictive density for the joint fit: "
      f"{int((fits['nll_all'] > fits['plugin_nll_all']).sum())} of {len(fits)}")

cells = fits.groupby(["dataset", "region"])[
    ["plugin_rmse_re", "rmse_re", "plugin_rmse_im", "rmse_im"]].mean().loc[utils.REGIONS]
cells.columns = pd.MultiIndex.from_tuples(
    [("Re", "two-step"), ("Re", "joint"), ("Im", "two-step"), ("Im", "joint")])
cells.index = [f"{d.split('_')[0]} {r}" for d, r in utils.REGIONS]
cells.map(utils.two_digits)

Runs with a higher training marginal likelihood for the joint fit: 31 of 31
States with a worse held-out log predictive density for the joint fit: 25 of 31


Re               Im         
            two-step   joint two-step    joint
D1 interior    0.042   0.049    0.026    0.028
D1 boundary   0.0078  0.0091   0.0081   0.0063
D2 interior     0.33    0.20    0.051    0.059
D2 boundary   0.0093   0.016   0.0065   0.0056
D3 interior   0.0025  0.0021  0.00044  0.00054
D3 boundary    0.014   0.014    0.042    0.044